# Chapter 28: Real Time vs Batch Systems

<a href="../lite/lab/index.html?path=ch28_realtime_vs_batch.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.optimize import least_squares
import time

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

A self driving car cannot wait until the end of the trip to figure out
where it is. It needs an answer NOW, every 10 milliseconds. But a
surveying drone can fly the whole mission, land, and spend 10 minutes
optimizing a perfect map. These two use cases lead to fundamentally
different architectures.

**Online (real time)** systems process measurements one at a time and
maintain a running estimate. **Offline (batch)** systems collect all
data first and solve a single large optimization problem. This chapter
builds both from scratch, runs them on identical data, and compares
accuracy, computation cost, and design tradeoffs.

```{admonition} What you will build
:class: tip

- Run the same estimation problem with both online (Kalman) and batch (least squares) approaches
- Prove that both give the same answer for linear Gaussian problems
- Compare computation time and accuracy tradeoffs
- Understand when to use real time filtering vs offline optimization

**Real world application:** Self driving cars need real time estimates. Surveying drones optimize offline. After this chapter, you will know which architecture to choose for your application.
```

## 28.1 Online Estimation: The Kalman Filter

The Kalman filter processes one measurement at a time. At each step $k$:

**Predict:**
$$\hat{\mathbf{x}}_{k|k-1} = \mathbf{F} \hat{\mathbf{x}}_{k-1|k-1}$$
$$\mathbf{P}_{k|k-1} = \mathbf{F} \mathbf{P}_{k-1|k-1} \mathbf{F}^T + \mathbf{Q}$$

**Update:**
$$\mathbf{K}_k = \mathbf{P}_{k|k-1} \mathbf{H}^T (\mathbf{H} \mathbf{P}_{k|k-1} \mathbf{H}^T + \mathbf{R})^{-1}$$
$$\hat{\mathbf{x}}_{k|k} = \hat{\mathbf{x}}_{k|k-1} + \mathbf{K}_k (\mathbf{z}_k - \mathbf{H} \hat{\mathbf{x}}_{k|k-1})$$

The key property: **constant computation per step**, regardless of how
many measurements have come before.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
n_steps = 50                  # number of time steps
dt = 1.0                      # time step
sigma_process = 0.5           # process noise std
sigma_meas = 1.0              # measurement noise std
# ─────────────────────────────────────────────────────────────────────────────

# Ground truth: 2D position with sinusoidal velocity
gt_x = np.zeros((n_steps, 2))
gt_v = np.zeros((n_steps, 2))
for k in range(n_steps):
    t = k * dt
    gt_x[k] = [5.0 * np.sin(0.1 * t), 3.0 * np.cos(0.15 * t)]
    gt_v[k] = [5.0 * 0.1 * np.cos(0.1 * t), -3.0 * 0.15 * np.sin(0.15 * t)]

# Generate noisy measurements of position
measurements = gt_x + np.random.randn(n_steps, 2) * sigma_meas

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(gt_x[:, 0], gt_x[:, 1], 'forestgreen', lw=2, label='Ground truth')
ax.scatter(measurements[:, 0], measurements[:, 1], c='tomato', s=20,
           alpha=0.5, label='Noisy measurements')
ax.set_xlabel('x (m)', fontsize=12); ax.set_ylabel('y (m)', fontsize=12)
ax.set_title('Tracking problem: estimate position from noisy data', fontsize=13)
ax.legend(fontsize=11); ax.set_aspect('equal')
plt.tight_layout(); plt.show()
print(f'{n_steps} time steps, measurement noise = {sigma_meas} m')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
kf_q_scale = 1.0             # scale process noise covariance
# ─────────────────────────────────────────────────────────────────────────────

# Kalman filter: state = [x, y, vx, vy]
F = np.array([[1, 0, dt, 0],
              [0, 1, 0, dt],
              [0, 0, 1, 0],
              [0, 0, 0, 1]], dtype=float)

H = np.array([[1, 0, 0, 0],
              [0, 1, 0, 0]], dtype=float)

Q = np.diag([0.01, 0.01, sigma_process**2, sigma_process**2]) * kf_q_scale
R = np.eye(2) * sigma_meas**2

# Run Kalman filter and time each step
kf_states = np.zeros((n_steps, 4))
kf_covs = np.zeros((n_steps, 4, 4))
kf_times = np.zeros(n_steps)

x_kf = np.array([measurements[0, 0], measurements[0, 1], 0.0, 0.0])
P_kf = np.diag([sigma_meas**2, sigma_meas**2, 1.0, 1.0])

for k in range(n_steps):
    t_start = time.perf_counter()
    
    if k > 0:
        # Predict
        x_kf = F @ x_kf
        P_kf = F @ P_kf @ F.T + Q
    
    # Update with measurement
    z = measurements[k]
    y_inn = z - H @ x_kf
    S = H @ P_kf @ H.T + R
    K = P_kf @ H.T @ np.linalg.inv(S)
    x_kf = x_kf + K @ y_inn
    P_kf = (np.eye(4) - K @ H) @ P_kf
    
    kf_times[k] = time.perf_counter() - t_start
    kf_states[k] = x_kf
    kf_covs[k] = P_kf

kf_errors = np.linalg.norm(kf_states[:, :2] - gt_x, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(gt_x[:, 0], gt_x[:, 1], 'forestgreen', lw=2, label='Truth')
ax.plot(kf_states[:, 0], kf_states[:, 1], 'steelblue', lw=2,
        marker='o', ms=3, label='KF estimate')
ax.scatter(measurements[:, 0], measurements[:, 1], c='tomato', s=15, alpha=0.3)
ax.set_aspect('equal'); ax.legend(fontsize=10)
ax.set_title('Kalman filter tracks the target', fontsize=12)

ax = axes[1]
ax.plot(kf_errors, 'steelblue', lw=2)
ax.set_xlabel('Time step', fontsize=12)
ax.set_ylabel('Position error (m)', fontsize=12)
ax.set_title('KF error over time (converges quickly)', fontsize=12)

plt.tight_layout(); plt.show()
print(f'KF mean error: {np.mean(kf_errors):.4f} m')
print(f'KF mean computation time: {np.mean(kf_times)*1e6:.1f} microseconds/step')

In [ ]:
# Show the estimate evolving step by step with uncertainty ellipses
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
show_steps = [1, 5, 10, 20, 35, 49]

for idx, step in enumerate(show_steps):
    ax = axes[idx // 3, idx % 3]
    ax.plot(gt_x[:step+1, 0], gt_x[:step+1, 1], 'forestgreen', lw=2, alpha=0.5)
    ax.scatter(measurements[:step+1, 0], measurements[:step+1, 1],
              c='tomato', s=15, alpha=0.3)
    ax.plot(kf_states[:step+1, 0], kf_states[:step+1, 1], 'steelblue',
            lw=2, marker='o', ms=3)
    
    # Draw 2 sigma covariance ellipse at current position
    cov2d = kf_covs[step, :2, :2]
    vals, vecs = np.linalg.eigh(cov2d)
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    w, h = 2 * 2 * np.sqrt(np.maximum(vals, 0))
    ell = Ellipse(xy=kf_states[step, :2], width=w, height=h,
                  angle=angle, fill=False, color='orange', lw=2)
    ax.add_patch(ell)
    
    ax.set_title(f'Step {step}: error = {kf_errors[step]:.3f} m', fontsize=11)
    ax.set_xlim(-7, 7); ax.set_ylim(-5, 5)
    ax.set_aspect('equal')

plt.suptitle('Kalman filter evolving over time (orange = 2 sigma uncertainty)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

**Observation:** The KF uncertainty shrinks rapidly in the first few
steps as the filter "locks on" to the target. After that, uncertainty
reaches a steady state determined by the balance between process noise
(which grows uncertainty) and measurements (which shrink it). The
computation per step is constant: the same matrix operations regardless
of whether it is step 2 or step 2000.

## 28.2 Offline Optimization: Batch Least Squares

The batch approach processes **all measurements at once**. We stack all
unknowns (positions at every time step) into a single vector and solve
a large least squares problem:

$$\min_{\mathbf{x}_{0:K}} \sum_{k=1}^{K} \|\mathbf{x}_k - \mathbf{F}\mathbf{x}_{k-1}\|^2_{\mathbf{Q}^{-1}} + \sum_{k=0}^{K} \|\mathbf{z}_k - \mathbf{H}\mathbf{x}_k\|^2_{\mathbf{R}^{-1}}$$

This considers all data simultaneously, finding the globally optimal
trajectory.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
batch_method = 'lm'          # scipy least_squares method
# ─────────────────────────────────────────────────────────────────────────────

def batch_optimize(meas, n_s, dt_val, sig_proc, sig_meas, F_mat=None):
    """Batch least squares: optimize all positions at once."""
    if F_mat is None:
        F_mat = np.array([[1, 0, dt_val, 0],
                          [0, 1, 0, dt_val],
                          [0, 0, 1, 0],
                          [0, 0, 0, 1]], dtype=float)
    dim = 4 * n_s
    
    def residuals(x_flat):
        x = x_flat.reshape(n_s, 4)
        res = []
        # Measurement residuals
        for k in range(n_s):
            r = (x[k, :2] - meas[k]) / sig_meas
            res.extend(r)
        # Dynamics residuals
        for k in range(1, n_s):
            pred = F_mat @ x[k - 1]
            r_dyn = (x[k] - pred)
            r_dyn[:2] /= 0.1
            r_dyn[2:] /= sig_proc
            res.extend(r_dyn)
        return np.array(res)
    
    # Initialize with measurements (estimate velocity from diffs)
    x0 = np.zeros(dim)
    for k in range(n_s):
        x0[4*k:4*k+2] = meas[k]
        if k > 0:
            x0[4*k+2:4*k+4] = (meas[k] - meas[k-1]) / dt_val
    
    t_start = time.perf_counter()
    result = least_squares(residuals, x0, method=batch_method,
                           max_nfev=500 * dim)
    t_total = time.perf_counter() - t_start
    
    opt_states = result.x.reshape(n_s, 4)
    return opt_states, t_total

batch_states, batch_time = batch_optimize(
    measurements, n_steps, dt, sigma_process, sigma_meas, F)

batch_errors = np.linalg.norm(batch_states[:, :2] - gt_x, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(gt_x[:, 0], gt_x[:, 1], 'forestgreen', lw=2, label='Truth')
ax.plot(batch_states[:, 0], batch_states[:, 1], 'orange', lw=2,
        marker='s', ms=3, label='Batch estimate')
ax.scatter(measurements[:, 0], measurements[:, 1], c='tomato', s=15, alpha=0.3)
ax.set_aspect('equal'); ax.legend(fontsize=10)
ax.set_title('Batch least squares solution', fontsize=12)

ax = axes[1]
ax.plot(batch_errors, 'orange', lw=2)
ax.set_xlabel('Time step', fontsize=12)
ax.set_ylabel('Position error (m)', fontsize=12)
ax.set_title('Batch error over time', fontsize=12)

plt.tight_layout(); plt.show()
print(f'Batch mean error: {np.mean(batch_errors):.4f} m')
print(f'Batch total computation time: {batch_time*1000:.1f} ms')

In [ ]:
# Batch uses future data to refine past estimates. Show this.
# Run batch on partial data: first 10, 20, 30, 40, 50 steps
partial_sizes = [10, 20, 30, 40, 50]
partial_errors_at_5 = []

for ns in partial_sizes:
    ps, _ = batch_optimize(measurements[:ns], ns, dt, sigma_process,
                            sigma_meas, F)
    partial_errors_at_5.append(np.linalg.norm(ps[5, :2] - gt_x[5]))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar([str(s) for s in partial_sizes], partial_errors_at_5,
       color='orange', alpha=0.8)
ax.axhline(kf_errors[5], color='steelblue', lw=2, ls='--',
           label=f'KF error at step 5 = {kf_errors[5]:.3f} m')
ax.set_xlabel('Data available (steps)', fontsize=12)
ax.set_ylabel('Error at step 5 (m)', fontsize=12)
ax.set_title('Batch estimate of step 5 improves with more future data', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

print('More data (including future measurements) lets batch refine past estimates.')
print('The KF cannot do this because it only processes forward.')

**Key difference:** The batch optimizer has access to **future data**.
At step 5, the KF has only seen measurements 0 through 5. The batch
optimizer has seen measurements 0 through 49. Future measurements
constrain the velocity estimate at step 5, which in turn constrains
the position. This "smoothing" effect is the fundamental advantage of
batch methods.

## 28.3 Tradeoffs: Side by Side Comparison

| Property | Online (KF) | Batch (LS) |
|----------|-------------|------------|
| **When available** | After each measurement | After all data collected |
| **Computation per step** | $O(d^3)$, constant | N/A (one big solve) |
| **Total computation** | $O(K \cdot d^3)$ | $O(K^2 \cdot d^3)$ or more |
| **Memory** | $O(d^2)$ (current state + cov) | $O(K \cdot d)$ (all states) |
| **Accuracy** | Good, improves over time | Optimal (uses future data) |
| **Can revise past?** | No (forward only) | Yes (full trajectory) |

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
highlight_step = 25           # step to highlight in comparison
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Trajectory comparison
ax = axes[0]
ax.plot(gt_x[:, 0], gt_x[:, 1], 'forestgreen', lw=2, label='Truth')
ax.plot(kf_states[:, 0], kf_states[:, 1], 'steelblue', lw=2,
        marker='o', ms=3, label='Online (KF)')
ax.plot(batch_states[:, 0], batch_states[:, 1], 'orange', lw=2,
        marker='s', ms=3, label='Batch (LS)')
ax.scatter(measurements[:, 0], measurements[:, 1], c='tomato', s=10, alpha=0.2)
ax.set_aspect('equal'); ax.legend(fontsize=9)
ax.set_title('Both methods track the target', fontsize=12)

# Error comparison
ax = axes[1]
ax.plot(kf_errors, 'steelblue', lw=2, label='Online (KF)')
ax.plot(batch_errors, 'orange', lw=2, label='Batch (LS)')
ax.axvline(highlight_step, color='gray', ls=':', lw=1.5)
ax.set_xlabel('Time step', fontsize=12)
ax.set_ylabel('Position error (m)', fontsize=12)
ax.set_title('Error comparison at each time step', fontsize=12)
ax.legend(fontsize=10)

# Computation time
ax = axes[2]
kf_cumtime = np.cumsum(kf_times)
ax.plot(kf_cumtime * 1e6, 'steelblue', lw=2, label='Online (cumulative)')
ax.axhline(batch_time * 1e6, color='orange', lw=2, ls='--',
           label=f'Batch total ({batch_time*1e6:.0f} us)')
ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('Computation time (us)', fontsize=12)
ax.set_title('Computation cost', fontsize=12)
ax.legend(fontsize=9)

plt.suptitle('Online vs Batch: same data, different architectures',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

print(f'\nAt step {highlight_step}:')
print(f'  KF error:    {kf_errors[highlight_step]:.4f} m (available immediately)')
print(f'  Batch error: {batch_errors[highlight_step]:.4f} m (available only after all data)')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
test_sizes = [10, 20, 30, 50, 75, 100]
# ─────────────────────────────────────────────────────────────────────────────

# Scaling: how does computation time grow with problem size?
kf_total_times = []
batch_total_times = []

for ns in test_sizes:
    # Generate data for this size
    gt_test = np.zeros((ns, 2))
    for k in range(ns):
        t = k * dt
        gt_test[k] = [5.0 * np.sin(0.1 * t), 3.0 * np.cos(0.15 * t)]
    meas_test = gt_test + np.random.randn(ns, 2) * sigma_meas
    
    # KF timing
    t0 = time.perf_counter()
    x_t = np.array([meas_test[0, 0], meas_test[0, 1], 0.0, 0.0])
    P_t = np.diag([sigma_meas**2, sigma_meas**2, 1.0, 1.0])
    for k in range(ns):
        if k > 0:
            x_t = F @ x_t
            P_t = F @ P_t @ F.T + Q
        z_t = meas_test[k]
        S_t = H @ P_t @ H.T + R
        K_t = P_t @ H.T @ np.linalg.inv(S_t)
        x_t = x_t + K_t @ (z_t - H @ x_t)
        P_t = (np.eye(4) - K_t @ H) @ P_t
    kf_total_times.append(time.perf_counter() - t0)
    
    # Batch timing
    _, bt = batch_optimize(meas_test, ns, dt, sigma_process, sigma_meas, F)
    batch_total_times.append(bt)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(test_sizes, np.array(kf_total_times) * 1000, 'steelblue', lw=2,
        marker='o', ms=8, label='Online (KF)')
ax.plot(test_sizes, np.array(batch_total_times) * 1000, 'orange', lw=2,
        marker='s', ms=8, label='Batch (LS)')
ax.set_xlabel('Number of time steps', fontsize=12)
ax.set_ylabel('Total computation time (ms)', fontsize=12)
ax.set_title('Scaling: KF is linear, batch grows superlinearly', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout(); plt.show()

print('KF scales linearly: O(N).')
print('Batch scales superlinearly: O(N * iterations * N).')

**Summary of tradeoffs:**

- **Use online (KF)** when you need an estimate immediately at each
  step, latency matters, and the problem is moderate size.
- **Use batch (LS)** when you can afford to wait, you want maximum
  accuracy, and you want to refine past estimates with future data.
- **Hybrid approaches** (sliding window, incremental smoothing) combine
  the best of both worlds.

---

## Capstone: EKF vs Periodic Graph Optimization

A robot drives for 100 steps. We compare two estimation strategies:

1. **EKF (online):** Updates the state at every step. Constant time.
2. **Graph optimization (batch, every 10 steps):** Collects 10 steps of
   data, then optimizes the full trajectory from the beginning.

We track which approach is more accurate at each moment.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(99)
n_cap = 100
batch_interval = 10           # run batch every N steps
sigma_proc_cap = 0.4
sigma_meas_cap = 1.2
# ─────────────────────────────────────────────────────────────────────────────

# Generate longer trajectory
gt_cap = np.zeros((n_cap, 2))
for k in range(n_cap):
    t = k * dt
    gt_cap[k] = [8.0 * np.sin(0.07 * t) + 0.02 * t,
                 5.0 * np.cos(0.11 * t)]
meas_cap = gt_cap + np.random.randn(n_cap, 2) * sigma_meas_cap

Q_cap = np.diag([0.01, 0.01, sigma_proc_cap**2, sigma_proc_cap**2])
R_cap = np.eye(2) * sigma_meas_cap**2

# Run EKF on all 100 steps
ekf_states = np.zeros((n_cap, 4))
ekf_errors = np.zeros(n_cap)
x_ekf = np.array([meas_cap[0, 0], meas_cap[0, 1], 0.0, 0.0])
P_ekf = np.diag([sigma_meas_cap**2, sigma_meas_cap**2, 2.0, 2.0])

for k in range(n_cap):
    if k > 0:
        x_ekf = F @ x_ekf
        P_ekf = F @ P_ekf @ F.T + Q_cap
    y_e = meas_cap[k] - H @ x_ekf
    S_e = H @ P_ekf @ H.T + R_cap
    K_e = P_ekf @ H.T @ np.linalg.inv(S_e)
    x_ekf = x_ekf + K_e @ y_e
    P_ekf = (np.eye(4) - K_e @ H) @ P_ekf
    ekf_states[k] = x_ekf
    ekf_errors[k] = np.linalg.norm(x_ekf[:2] - gt_cap[k])

print(f'EKF: updates at every step ({n_cap} updates)')
print(f'EKF mean error: {np.mean(ekf_errors):.4f} m')

In [ ]:
# Run periodic batch optimization
graph_states_full = ekf_states.copy()  # use KF between solves
graph_errors = np.full(n_cap, np.nan)
batch_solve_steps = []
batch_solve_times = []

for end_step in range(batch_interval, n_cap + 1, batch_interval):
    bs, bt = batch_optimize(meas_cap[:end_step], end_step, dt,
                             sigma_proc_cap, sigma_meas_cap, F)
    graph_states_full[:end_step] = bs
    batch_solve_steps.append(end_step - 1)
    batch_solve_times.append(bt)
    for k in range(end_step):
        graph_errors[k] = np.linalg.norm(bs[k, :2] - gt_cap[k])

print(f'Graph: batch solve every {batch_interval} steps '
      f'({len(batch_solve_steps)} solves)')
valid_graph = ~np.isnan(graph_errors)
print(f'Graph mean error (final): {np.mean(graph_errors[valid_graph]):.4f} m')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Trajectories
ax = axes[0, 0]
ax.plot(gt_cap[:, 0], gt_cap[:, 1], 'forestgreen', lw=2, label='Truth')
ax.plot(ekf_states[:, 0], ekf_states[:, 1], 'steelblue', lw=1.5,
        alpha=0.7, label='EKF (online)')
ax.plot(graph_states_full[:, 0], graph_states_full[:, 1], 'orange',
        lw=1.5, alpha=0.7, label='Graph (periodic batch)')
ax.set_aspect('equal'); ax.legend(fontsize=9)
ax.set_title('Both trajectories', fontsize=12)

# Error over time
ax = axes[0, 1]
ax.plot(ekf_errors, 'steelblue', lw=1.5, label='EKF (online)')
ax.plot(np.where(valid_graph)[0], graph_errors[valid_graph], 'orange',
        lw=1.5, label='Graph (after last batch)')
for s in batch_solve_steps:
    ax.axvline(s, color='orange', alpha=0.15, lw=1)
ax.set_xlabel('Step', fontsize=12)
ax.set_ylabel('Error (m)', fontsize=12)
ax.set_title('Error comparison (shaded lines = batch solves)', fontsize=12)
ax.legend(fontsize=9)

# Who wins at each step?
ax = axes[1, 0]
ekf_wins = np.zeros(n_cap)
graph_wins = np.zeros(n_cap)
for k in range(n_cap):
    if valid_graph[k]:
        if ekf_errors[k] < graph_errors[k]:
            ekf_wins[k] = 1
        else:
            graph_wins[k] = 1
ax.bar(range(n_cap), ekf_wins, color='steelblue', alpha=0.6, label='EKF wins')
ax.bar(range(n_cap), -graph_wins, color='orange', alpha=0.6, label='Graph wins')
ax.set_xlabel('Step', fontsize=12)
ax.set_title('Which method is more accurate at each step?', fontsize=12)
ax.legend(fontsize=9); ax.set_yticks([])

# Summary statistics
ax = axes[1, 1]
final_graph_err = graph_errors[valid_graph]
labels = ['Mean', 'Max', 'Std']
ekf_stats = [np.mean(ekf_errors), np.max(ekf_errors), np.std(ekf_errors)]
graph_st = [np.mean(final_graph_err), np.max(final_graph_err),
            np.std(final_graph_err)]

x_pos = np.arange(len(labels))
w = 0.35
ax.bar(x_pos - w/2, ekf_stats, w, color='steelblue', alpha=0.8, label='EKF')
ax.bar(x_pos + w/2, graph_st, w, color='orange', alpha=0.8, label='Graph')
ax.set_xticks(x_pos); ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel('Error (m)', fontsize=12)
ax.set_title('Summary statistics', fontsize=12)
ax.legend(fontsize=10)

plt.suptitle('Capstone: EKF vs Periodic Graph Optimization (100 steps)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Show how batch refines early estimates as more data comes in
fig, ax = plt.subplots(figsize=(10, 5))

err_at_5 = []
solve_labels = []
for end_step in range(batch_interval, n_cap + 1, batch_interval):
    bs, _ = batch_optimize(meas_cap[:end_step], end_step, dt,
                            sigma_proc_cap, sigma_meas_cap, F)
    err_at_5.append(np.linalg.norm(bs[5, :2] - gt_cap[5]))
    solve_labels.append(str(end_step))

ax.bar(solve_labels, err_at_5, color='orange', alpha=0.8)
ax.axhline(ekf_errors[5], color='steelblue', lw=2, ls='--',
           label=f'EKF estimate at step 5 = {ekf_errors[5]:.3f} m')
ax.set_xlabel('Steps of data available to batch', fontsize=12)
ax.set_ylabel('Error at step 5 (m)', fontsize=12)
ax.set_title('Batch smoothing: past estimates improve with future data', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

print('The batch optimizer refines past estimates as future data arrives.')
print('This smoothing effect is impossible with a forward only filter.')

In [ ]:
# Computation time comparison
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(solve_labels, np.array(batch_solve_times) * 1000,
       color='orange', alpha=0.8, label='Per batch solve')
ax.set_xlabel('Steps included', fontsize=12)
ax.set_ylabel('Batch solve time (ms)', fontsize=12)
ax.set_title('Batch computation grows with problem size', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

total_kf = np.sum(kf_times) * 1000
total_batch = np.sum(batch_solve_times) * 1000
print(f'Total EKF time: {total_kf:.2f} ms')
print(f'Total batch time: {total_batch:.2f} ms')
print(f'Batch is {total_batch / max(total_kf, 1e-9):.1f}x more expensive')

**Capstone observations:**

- The EKF provides estimates **immediately** at each step with constant
  computation cost. Its accuracy is limited because it only uses past
  and current measurements.
- The graph optimizer produces **more accurate estimates** because it
  uses all available data, including future measurements (smoothing).
  It introduces latency and higher computational cost.
- The periodic batch approach is a practical compromise: run the filter
  for real time estimates, then periodically re-optimize the full
  trajectory for a more accurate map.
- In practice, most modern SLAM systems use a **hybrid architecture**:
  a fast filter for tracking plus a periodic optimizer for mapping.

---

## Exercises

### Exercise 28.1: Rauch Tung Striebel Smoother

Implement the **RTS smoother**: run the Kalman filter forward, then a
backward pass that refines past estimates using future data. Compare
its accuracy to the forward only KF and the batch optimizer. The
backward pass equations are:

$$\mathbf{C}_k = \mathbf{P}_{k|k} \mathbf{F}^T \mathbf{P}_{k+1|k}^{-1}$$
$$\hat{\mathbf{x}}_{k|K} = \hat{\mathbf{x}}_{k|k} + \mathbf{C}_k (\hat{\mathbf{x}}_{k+1|K} - \hat{\mathbf{x}}_{k+1|k})$$

In [ ]:
# Your code here

### Exercise 28.2: Batch Interval Sweep

Vary the batch optimization interval from 5 to 50 steps. For each
interval, compute: (a) mean error over the trajectory, (b) total
computation time. Plot both. What is the sweet spot where you get
near batch accuracy with reasonable computation?

In [ ]:
# Your code here

### Exercise 28.3: Sliding Window Optimizer

Implement a **sliding window optimizer** that keeps only the last $W$
steps and optimizes over that window at each step. Compare accuracy
and timing vs. full batch and KF for $W = 5, 10, 20$.

In [ ]:
# Your code here

### Exercise 28.4: Measurement Dropout (challenge)

Simulate a sensor that drops measurements randomly (50% dropout rate).
Compare how the KF and batch optimizer handle missing data. Which
approach degrades more gracefully? Plot error vs. dropout rate from
0% to 80%.

In [ ]:
# Your code here